In this notebook we perform ALL-to-ALL model comparisons between models and monkey.

In [ ]:
# Resolve project inputs consistently when Jupyter opens this subdirectory.
import os

from paths import ROOT

os.chdir(ROOT)
# Import the numerical and plotting tools used for all-to-all neuron matching.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

In [ ]:
# Load the preserved condition/epoch response tables and align column names.
mk_df_all_pre = pd.read_pickle("artifacts/dpx/monkey/table_pre.pkl")
mk_df_all_pre["trialtype"] = "pre"
mk_df_all_bal = pd.read_pickle("artifacts/dpx/monkey/table_bal.pkl")
mk_df_all_bal["trialtype"] = "bal"
# align the naming
for df in [mk_df_all_bal, mk_df_all_pre]:
    new_cols = []
    for col in df.columns:
        if col.startswith("b_"):
            new_cols.append(f"before_{col.split('_')[1]}")
        elif col.startswith("c_"):
            new_cols.append(f"cue_{col.split('_')[1]}")
        elif col.startswith("d_"):
            new_cols.append(f"delay_{col.split('_')[1]}")
        elif col.startswith("p_"):
            new_cols.append(f"probe_{col.split('_')[1]}")
        elif col.startswith("r_"):
            new_cols.append(f"after_{col.split('_')[1]}")
        else:
            new_cols.append(col)
    df.columns = new_cols


mk_df_all = pd.concat([mk_df_all_bal, mk_df_all_pre])
mk_df_all["cue_probe_trialtype"] = mk_df_all["cue_probe"] + mk_df_all["trialtype"]


md_classical_df_rec_pre = pd.read_pickle("artifacts/dpx/classical/table_pre.pkl")
md_classical_df_rec_pre["cue_probe"] = (
    md_classical_df_rec_pre["cue"] + md_classical_df_rec_pre["probe"]
)
md_classical_df_rec_pre["trialtype"] = "pre"

md_classical_df_rec_bal = pd.read_pickle("artifacts/dpx/classical/table_bal.pkl")
md_classical_df_rec_bal["cue_probe"] = (
    md_classical_df_rec_bal["cue"] + md_classical_df_rec_bal["probe"]
)
md_classical_df_rec_bal["trialtype"] = "bal"

md_classical_df_all = pd.concat([md_classical_df_rec_bal, md_classical_df_rec_pre])
md_classical_df_all["cue_probe_trialtype"] = (
    md_classical_df_all["cue_probe"] + md_classical_df_all["trialtype"]
)


md_df_rec_pre = pd.read_pickle("artifacts/dpx/predictive/table_pre.pkl")
md_df_rec_pre["cue_probe"] = md_df_rec_pre["cue"] + md_df_rec_pre["probe"]
md_df_rec_pre["trialtype"] = "pre"


md_df_rec_bal = pd.read_pickle("artifacts/dpx/predictive/table_bal.pkl")
md_df_rec_bal["cue_probe"] = md_df_rec_bal["cue"] + md_df_rec_bal["probe"]
md_df_rec_bal["trialtype"] = "bal"

md_df_all = pd.concat([md_df_rec_bal, md_df_rec_pre])
md_df_all["cue_probe_trialtype"] = md_df_all["cue_probe"] + md_df_all["trialtype"]

In [ ]:
# Correlate each monkey/model pair across the existing condition and epoch features.
def cal_full_corr_matrix(dfA, dfB):
    nidsA = []
    for col in dfA.columns:
        if col.startswith("before_"):
            nidsA.append(int(col.split("_")[1]))

    nidsB = []
    for col in dfB.columns:
        if col.startswith("before_"):
            nidsB.append(int(col.split("_")[1]))

    corr_info_matrix = np.zeros((len(nidsA), len(nidsB), 2))
    condition_key = "cue_probe_trialtype"
    for i, nid_i in enumerate(nidsA):
        dfA_slice = dfA[
            [
                condition_key,
                f"before_{nid_i}",
                f"cue_{nid_i}",
                f"delay_{nid_i}",
                f"probe_{nid_i}",
                f"after_{nid_i}",
            ]
        ]
        dfA_slice_mean = dfA_slice.groupby(condition_key).mean()
        dfA_slice_mean.sort_index(inplace=True)

        for j, nid_j in enumerate(nidsB):
            dfB_slice = dfB[
                [
                    condition_key,
                    f"before_{nid_j}",
                    f"cue_{nid_j}",
                    f"delay_{nid_j}",
                    f"probe_{nid_j}",
                    f"after_{nid_j}",
                ]
            ]
            dfB_slice_mean = dfB_slice.groupby(condition_key).mean()
            dfB_slice_mean.sort_index(inplace=True)

            corr = stats.pearsonr(dfA_slice_mean.values.flatten(), dfB_slice_mean.values.flatten())

            corr_info_matrix[i, j, 0] = corr.statistic
            corr_info_matrix[i, j, 1] = corr.pvalue

    return corr_info_matrix

for each neuron in either monkey/model, we treat its condition-phase combinations as variables.
we have four possible conditions and 5 phases, so in total we have 20 observations.

In [ ]:
# Compute predictive and classical matching matrices with the same criterion.
cim_model = cal_full_corr_matrix(mk_df_all, md_df_all)
cim_classical_model = cal_full_corr_matrix(mk_df_all, md_classical_df_all)

In [ ]:
# Apply the original p-value threshold to both model comparison matrices.
p_threshold = 0.001
print(f"Ratio of neuron pairs whose correlation p-value < {p_threshold}")
print("Our model:", (cim_model[:, :, 1] < p_threshold).mean())
print("Classical RNN:", (cim_classical_model[:, :, 1] < p_threshold).mean())

effective_cim_model = cim_model[:, :, 0] * (cim_model[:, :, 1] < p_threshold)
effective_cim_classical_model = cim_classical_model[:, :, 0] * (
    cim_classical_model[:, :, 1] < p_threshold
)

In [ ]:
# Build the shared comparison arrays used by the manuscript-style figure.
# The heatmaps keep the original thresholded all-to-all correlations, while the summary panels quantify pair frequency and strength.
sig_model = cim_model[:, :, 1] < p_threshold
sig_classical_model = cim_classical_model[:, :, 1] < p_threshold
abs_cim_model = np.abs(cim_model[:, :, 0] * sig_model)
abs_cim_classical_model = np.abs(cim_classical_model[:, :, 0] * sig_classical_model)
nonzero_corr_cim_model = abs_cim_model.flatten()[abs_cim_model.flatten() > 0]
nonzero_corr_cim_classical_model = abs_cim_classical_model.flatten()[
    abs_cim_classical_model.flatten() > 0
]
ratio_model = sig_model.mean() * 100
ratio_classical_model = sig_classical_model.mean() * 100
u, p = stats.mannwhitneyu(
    nonzero_corr_cim_model,
    nonzero_corr_cim_classical_model,
    alternative="greater",
)
median_model = np.median(nonzero_corr_cim_model)
median_classical_model = np.median(nonzero_corr_cim_classical_model)
median_delta = median_model - median_classical_model
sig_count_model = sig_model.sum(axis=1)
sig_count_classical_model = sig_classical_model.sum(axis=1)
count_u, count_p = stats.wilcoxon(sig_count_model, sig_count_classical_model, alternative="greater")

# Use model-specific colors rather than the AX/AY/BX/BY condition palette.
# Vermilion highlights the predictive-coding model; neutral gray keeps the classical baseline visually secondary.
model_color = "#D55E00"
classical_color = "#8A8A8A"
model_light = "#F2B38C"
classical_light = "#CFCFCF"

# Build one compact multi-panel figure in the same manuscript style as make_figures.ipynb.
# The figure reads left-to-right: landscape, pair frequency, pair strength, and per-monkey repertoire size.
fig, axes = plt.subplots(
    2,
    3,
    figsize=(9, 4),
    width_ratios=[2.2, 1.0, 1.25],
)

# Plot all-to-all significant matching landscapes for the predictive-coding model and classical baseline.
# Both heatmaps use the same |r| scale so density and strength can be compared directly.
axes[0, 0].imshow(
    abs_cim_model,
    vmin=0,
    vmax=1,
    cmap=sns.light_palette(model_color, as_cmap=True),
    aspect="auto",
)
axes[0, 0].set_title("Predictive coding")
axes[0, 0].set_ylabel("Monkey neurons")
axes[0, 0].set_xticklabels([])
axes[1, 0].imshow(
    abs_cim_classical_model,
    vmin=0,
    vmax=1,
    cmap=sns.light_palette(classical_color, as_cmap=True),
    aspect="auto",
)
axes[1, 0].set_title("Classical RNN")
axes[1, 0].set_ylabel("Monkey neurons")
axes[1, 0].set_xlabel("Model neurons")

# Plot the significant pair ratio as a direct count-style summary of the heatmaps.
# This panel supports the claim that the predictive-coding model contains more monkey-like pairs.
ratio_df = pd.DataFrame(
    {
        "Model": ["PC RNN", "Classical"],
        "Matched pairs (%)": [ratio_model, ratio_classical_model],
    }
)
sns.barplot(
    data=ratio_df,
    x="Model",
    y="Matched pairs (%)",
    ax=axes[0, 1],
    palette=[model_color, classical_color],
)
axes[0, 1].set_title("Matched pairs")
axes[0, 1].set_xlabel("")
axes[0, 1].set_ylim(0, max(ratio_df["Matched pairs (%)"]) * 1.25)
for idx, value in enumerate(ratio_df["Matched pairs (%)"]):
    axes[0, 1].text(idx, value, f"{value:.1f}%", ha="center", va="bottom", fontsize=10)

# Plot the survival curve of significant |r| values so higher curves mean stronger matched pairs.
# Dashed lines mark medians, and the title reports the same Mann-Whitney test as the original notebook.
sorted_model = np.sort(nonzero_corr_cim_model)
sorted_classical = np.sort(nonzero_corr_cim_classical_model)
y_model = np.arange(len(sorted_model), 0, -1) / len(sorted_model)
y_classical = np.arange(len(sorted_classical), 0, -1) / len(sorted_classical)
axes[1, 1].step(sorted_model, y_model, where="post", color=model_color, label="PC RNN")
axes[1, 1].step(
    sorted_classical, y_classical, where="post", color=classical_color, label="Classical"
)
axes[1, 1].axvline(median_model, color=model_color, linestyle="--", linewidth=1)
axes[1, 1].axvline(median_classical_model, color=classical_color, linestyle="--", linewidth=1)
axes[1, 1].set_title(f"Matched strength\np={p:.1e}")
axes[1, 1].set_xlabel(f"|r| threshold (Delta med={median_delta:.3f})")
axes[1, 1].set_ylabel("Fraction >= threshold")
axes[1, 1].legend(frameon=False, loc="upper right", handlelength=1.0, fontsize=9)

# Plot per-monkey-neuron matched model-neuron counts as a paired summary.
# This avoids relying only on all-to-all pair counts and asks whether each monkey unit has a broader PC-RNN repertoire.
x_positions = np.array([0, 1])
for pc_count, classical_count in zip(sig_count_model, sig_count_classical_model):
    axes[0, 2].plot(x_positions, [pc_count, classical_count], color="lightgray", linewidth=0.8)
axes[0, 2].scatter(np.zeros_like(sig_count_model), sig_count_model, color=model_color, s=16)
axes[0, 2].scatter(
    np.ones_like(sig_count_classical_model), sig_count_classical_model, color=classical_color, s=16
)
axes[0, 2].set_xticks(x_positions)
axes[0, 2].set_xticklabels(["PC RNN", "Classical"], fontsize=9)
axes[0, 2].set_ylabel("Matched model neurons", fontsize=10)
axes[0, 2].set_title(f"Per-monkey count\np={count_p:.1e}", fontsize=11)

# Show the paired count difference as a compact distribution that matches the count panel above.
# Values above zero mean that a monkey neuron matches more predictive-coding units than classical units.
count_delta = sig_count_model - sig_count_classical_model
axes[1, 2].hist(count_delta, bins=12, color=model_light)
axes[1, 2].axvline(0, color="gray", linestyle="--", linewidth=1)
axes[1, 2].set_xlabel("PC - Classical", fontsize=10)
axes[1, 2].set_ylabel("Count", fontsize=10)
axes[1, 2].set_title("Count gain", fontsize=11)

# Apply the same cleanup and output convention as make_figures.ipynb.
# The legacy filename is kept as an alias for any downstream manuscript workflow that already points to it.
fig.tight_layout()
sns.despine()
fig.savefig("artifacts/dpx/figures/comparisons.png", dpi=300)
plt.show()

# Print the numerical story after the figure so the notebook remains self-documenting.
# These values correspond to the labels used in the manuscript-style panels.
print(f"Matched-pair ratio: PC RNN={ratio_model:.3f}%, Classical={ratio_classical_model:.3f}%")
print(f"Median significant |r|: PC RNN={median_model:.3f}, Classical={median_classical_model:.3f}")
print(f"Mann-Whitney U p={p}")
print(f"Match-count Wilcoxon p={count_p}")